# Publication-Ready Layerwise Uncertainty Decomposition Analysis (3x3 Grid)

This notebook generates publication-ready layerwise ensembling depth figures formatted for academic papers:
- **Decoupled Metric Figures**: Aleatoric Uncertainty ($H_{\text{mix}}$) and Epistemic Uncertainty ($\Delta H$) are plotted in **separate dedicated $3 \times 3$ figures** to eliminate visual clutter and curve tangling.
- **Merged Column Legends**: Legends are placed outside the subplots as clean column headers above Row 0 (`bbox_to_anchor`), completely eliminating inside-axis legend overlap.
- **$3 \times 3$ Grid Layout**: 3 Datasets (`TruthfulQA`, `OpenBookQA`, `ScienceQA`) $\times$ 3 Model Families (`LLaMA`, `Qwen`, `Phi`).
- **Line Formats**: Clean continuous lines (**no marker dots**), solid lines for Aleatoric ($H_{\text{mix}}$), dashed lines for Difference Epistemic ($\Delta H$).
- **Precision Scope**: **Strictly Full-Precision (FP16 / BF16)** models (quantization runs excluded to avoid overlap).
- **Evaluation Standard**: Standard BLEURT correctness threshold $\tau = 0.1$ (`GLOBAL_CORR_THRESHOLD = 0.1`).
- **Paper Exports**: First displayed in the notebook, then automatically saved to `figures/` as vector PDFs and 300 DPI PNGs.


In [ ]:
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import glob
import pickle as pkl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns

# Matplotlib publication aesthetic settings
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 11.5
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 9.5
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['figure.dpi'] = 120
print('Libraries and plotting parameters initialized successfully.')


## 1. Directory Scanning & Full-Precision Run Discovery

Scans canonical results directories (`result/entropies_unified`, `src/experiments/result/entropies_unified`, etc.).
Filters **strictly full-precision models** (excluding 4-bit quantization runs) and organizes them by dataset and architecture family.


In [ ]:
# Configuration - Candidate directories prioritized by canonical location
CANDIDATES = [
    '../experiments/result/entropies_unified',
    'src/experiments/result/entropies_unified',
    './result/entropies_unified',
    '../../result/entropies_unified',
    'result/entropies_unified',
    'entropies_unified',
    'src/experiments',
    '../experiments',
    './src/experiments',
    '../experiments/result/entropies3',
    'src/experiments/result/entropies3',
    './result/entropies3',
    '../experiments/result/entropies2',
    'src/experiments/result/entropies2',
    './result/entropies2',
    'src',
    './src',
    '.'
]

valid_dirs = []
for c in CANDIDATES:
    if os.path.exists(c) and any(f.endswith('.pkl') for f in os.listdir(c)):
        abs_c = os.path.abspath(c)
        if abs_c not in valid_dirs:
            valid_dirs.append(abs_c)

DATA_DIR = valid_dirs[0] if valid_dirs else '.'
print(f"Primary Data Directory: {DATA_DIR}")
print(f"Candidate Directories Found: {len(valid_dirs)}")

# Project-wide BLEURT threshold standard
GLOBAL_CORR_THRESHOLD = 0.1
EVAL_LABEL_KEY = 'bleurt'
print(f"Standard Correctness Threshold: BLEURT >= {GLOBAL_CORR_THRESHOLD}")

# Helper to classify architecture family
def get_model_family(model_name):
    m = str(model_name).lower().replace('_', '-')
    if 'llama' in m:
        return 'LLaMA'
    elif 'qwen' in m:
        return 'Qwen'
    elif 'phi' in m:
        return 'Phi'
    elif 'gemma' in m:
        return 'Gemma'
    elif 'mistral' in m:
        return 'Mistral'
    return 'Other'

# Discover runs
runs = []
seen_run_keys = set()

for d_path in valid_dirs:
    d_files = glob.glob(os.path.join(d_path, "*.pkl"))
    entropy_files = [f for f in d_files if "_entropies_" in os.path.basename(f) or os.path.basename(f).startswith("sdlg_entropies_")]
    for f in sorted(entropy_files):
        base = os.path.basename(f)
        label_file = f.replace("_entropies_", "_label_").replace("sdlg_entropies_", "sdlg_label_")
        if not os.path.exists(label_file):
            if 'qwen-1.5b' in label_file: label_file = label_file.replace('qwen-1.5b', 'qwen2.5-1.5b')
            elif 'qwen2.5-1.5b' in label_file: label_file = label_file.replace('qwen2.5-1.5b', 'qwen-1.5b')
        question_file = f.replace("_entropies_", "_question_").replace("sdlg_entropies_", "sdlg_question_")
        parts = base.replace(".pkl", "").split("_")
        model_name = "unknown"
        dataset_name = "truthfulqa"
        
        # Quantization detection
        is_quantized = any(p.lower() == '4bit' or p.lower().endswith('4bit') for p in parts) or ('4bit' in base.lower())
        
        for p in parts:
            p_lower = p.lower()
            if any(m in p_lower for m in ['llama', 'qwen', 'phi', 'opt', 'mistral', 'gemma', 'deepseek']):
                model_name = p
            if any(d in p_lower for d in ['truth', 'sciq', 'openbook', 'open', 'science', 'scienceqa', 'ambig', 'asqa', 'fresh', 'freshqa']):
                dataset_name = 'scienceqa' if 'science' in p_lower or 'sciq' in p_lower else ('openbookqa' if 'open' in p_lower else ('truthfulqa' if 'truth' in p_lower else p_lower))
                
        # ONLY collect strictly Full-Precision models
        if is_quantized:
            continue
            
        if os.path.exists(label_file):
            family = get_model_family(model_name)
            run_key = (dataset_name.lower(), model_name.lower())
            if run_key not in seen_run_keys:
                seen_run_keys.add(run_key)
                runs.append({
                    'name': f"{model_name} on {dataset_name}",
                    'model': model_name,
                    'family': family,
                    'dataset': dataset_name,
                    'entropy_file': f,
                    'label_file': label_file,
                    'question_file': question_file if os.path.exists(question_file) else None,
                    'dir': d_path
                })

print(f"\n[+] Discovered {len(runs)} Full-Precision runs across datasets:")
for r in runs:
    print(f"  - [{r['dataset'].upper():11}] {r['family']:6} | {r['model']} (from {os.path.basename(r['dir'])})")


## 2. Layerwise Metric Extraction Engine (BLEURT $\geq 0.1$)

Loads sample predictions and extracts AUROC curves across layer ensembling depths $K_a$:
- **Aleatoric Uncertainty ($H_{\text{mix}}$)**: Premature ensemble mixture entropy across depth $K_a$.
- **Difference Epistemic Uncertainty ($\Delta H$)**: Difference epistemic uncertainty $\Delta H = H_{\text{final}} - H_{\text{mix}}$ across depth $K_a$.


In [ ]:
def safe_extract_label(val, key='bleurt'):
    while isinstance(val, (list, tuple)) and len(val) > 0: val = val[0]
    if isinstance(val, dict):
        if key in val: val = val[key]
        elif 'correctness_dict' in val: val = val['correctness_dict'].get(key, 0.0)
        else:
            for k in ['bleurt_score', 'bleurt', 'rougeL', 'rouge1', 'exact_match']:
                if k in val: val = val[k]; break
            else: val = 0.0
    try:
        fval = float(val)
        return fval if np.isfinite(fval) else 0.0
    except Exception:
        return 0.0

def calculate_auroc(y_true, scores):
    mask = np.isfinite(y_true) & np.isfinite(scores)
    y_t, sc = y_true[mask], scores[mask]
    if len(np.unique(y_t)) < 2:
        return np.nan
    # Error detection AUROC: positive class is incorrect prediction (y_true == 0)
    y_err = (y_t == 0).astype(int)
    try:
        return float(roc_auc_score(y_err, sc))
    except Exception:
        return np.nan

def load_run_layer_data(run):
    with open(run['entropy_file'], 'rb') as f: raw_entropies = pkl.load(f)
    with open(run['label_file'], 'rb') as f: raw_labels = pkl.load(f)
    
    labels = [safe_extract_label(l, EVAL_LABEL_KEY) for l in raw_labels]
    num_instances = len(labels)
    is_correct = (pd.Series(labels) >= GLOBAL_CORR_THRESHOLD).astype(int).values
    
    chunk = raw_entropies[:num_instances] if isinstance(raw_entropies, list) and len(raw_entropies) >= num_instances else raw_entropies
    
    # Final entropy
    h_final = np.array([d.get('entropy_final', np.nan) if isinstance(d, dict) else np.nan for d in chunk])
    if np.isnan(h_final).all():
        h_final = np.array([d.get('entropy', np.nan) if isinstance(d, dict) else np.nan for d in chunk])
        
    # Extract premature mixture keys
    sample_d = chunk[0] if len(chunk) > 0 and isinstance(chunk[0], dict) else {}
    em_keys = [k for k in sample_d.keys() if k.startswith('entropy_mixture_') and k.replace('entropy_mixture_', '').replace('top_', '').isdigit()]
    
    def extract_depth(k):
        rem = k.replace('entropy_mixture_', '').replace('top_', '')
        return int(rem) if rem.isdigit() else 999
    em_keys.sort(key=extract_depth)
    
    depths, aurocs_em, aurocs_diff = [], [], []
    for k in em_keys:
        d_val = extract_depth(k) + 1
        if d_val > 64: continue
        
        h_mix = np.array([d.get(k, np.nan) if isinstance(d, dict) else np.nan for d in chunk])
        diff_h = h_final - h_mix
        
        auc_em = calculate_auroc(is_correct, h_mix)
        auc_diff = calculate_auroc(is_correct, diff_h)
        
        if np.isfinite(auc_em) and np.isfinite(auc_diff):
            depths.append(d_val)
            aurocs_em.append(auc_em)
            aurocs_diff.append(auc_diff)
            
    return {
        'model': run['model'],
        'family': run['family'],
        'dataset': run['dataset'],
        'accuracy': float(np.mean(is_correct)),
        'depths': depths,
        'aurocs_em': aurocs_em,
        'aurocs_diff': aurocs_diff
    }

sweep_results = []
for r in runs:
    try:
        res = load_run_layer_data(r)
        if res['depths']:
            sweep_results.append(res)
            print(f"  [OK] Processed {r['dataset'].upper():11} | {r['model']:22} ({len(res['depths'])} depths)")
    except Exception as e:
        print(f"  [X] Failed {r['model']}: {e}")

print(f"\nSuccessfully cached layerwise curves for {len(sweep_results)} full-precision runs.")


## 3. Publication $3 \times 3$ Grid Visualization & Paper Export

### Shared Plotting Engine & Merged Legend Architecture
To solve legend overlapping issues:
- **Zero In-Axes Legend Boxes**: Inside each subplot, curves have 100% unobstructed canvas.
- **Merged Column Legend Headers**: A single horizontal legend per model family is placed **outside the grid** directly above the top row (`bbox_to_anchor=(0.5, 1.05)`).
- **Clean Lines (No Dots)**: Uses smooth continuous lines without markers.
- **Separate Figures**: Generates **Aleatoric ($H_{\text{mix}}$)** and **Epistemic ($\Delta H$)** in two dedicated standalone figures.


In [ ]:
TARGET_DATASETS = [
    ('truthfulqa', 'TruthfulQA'),
    ('openbookqa', 'OpenBookQA'),
    ('scienceqa',  'ScienceQA')
]

TARGET_FAMILIES = ['LLaMA', 'Qwen', 'Phi']

FAMILY_TITLES = {
    'LLaMA': 'LLaMA Family',
    'Qwen':  'Qwen-2.5 Family',
    'Phi':   'Phi-3 Family'
}

# Consistent color assignment per model
MODEL_COLORS = {
    'llama-3.2-1b':      '#1f77b4',  # Blue
    'llama-3.2-3b':      '#2ca02c',  # Green
    'meta-llama-3-8b':   '#d62728',  # Red
    'llama-3-8b':        '#d62728',
    'qwen-1.5b':         '#9467bd',  # Purple
    'qwen2.5-1.5b':      '#9467bd',
    'qwen2.5-3b':        '#ff7f0e',  # Orange
    'qwen2.5-7b':        '#17becf',  # Cyan
    'qwen2.5-14b':       '#8c564b',  # Brown
    'qwen1.5-moe-a2.7b': '#e377c2',  # Pink
    'phi-3-mini':        '#e377c2',  # Pink
    'phi-3-mini-4k':     '#e377c2',
    'phi-3-medium-4k':   '#bcbd22',  # Yellow-Green
    'phi-3-medium':      '#bcbd22'
}

FALLBACK_PALETTE = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

def clean_model_label(m_str):
    """Concise labels per family preventing legend box collisions."""
    m = str(m_str).lower()
    if '1b' in m and 'llama' in m:
        return '3.2-1B'
    if '3b' in m and 'llama' in m:
        return '3.2-3B'
    if ('8b' in m or 'llama-3-8b' in m) and 'llama' in m:
        return '3-8B'
    
    if '1.5b' in m:
        return '1.5B'
    if '3b' in m and 'qwen' in m:
        return '3B'
    if '7b' in m and 'qwen' in m:
        return '7B'
    if '14b' in m and 'qwen' in m:
        return '14B'
        
    if 'mini' in m:
        return 'Mini'
    if 'medium' in m:
        return 'Medium'
        
    return str(m_str).replace('meta-', '').replace('-instruct', '').replace('-chat', '').replace('-4k', '').upper()

def model_sort_key(item):
    m = item['model'].lower()
    if '1b' in m or '1.5b' in m: return 1
    if '3b' in m or 'mini' in m: return 3
    if '7b' in m or '8b' in m: return 7
    if '14b' in m or 'medium' in m: return 14
    return 99

def plot_layerwise_unified_3x6(sweep_data, figsize=(18.0, 7.8)):
    """
    Unified publication 3x6 grid: combines Aleatoric and Epistemic side-by-side.
    - Dataset labels shifted slightly left closer to the plot boundary (x=1.018).
    - Cushioned horizontal margin between Y-axis label text and Column 0.
    - Generous vertical separation between column headers and legend boxes (pad=36, top=0.875).
    - Prominent, large legend font (fontsize=10.2).
    - Updated titles: 'Qwen-2.5 Family' and 'Phi-3 Family'.
    - Extreme left Y-axis ticks only (zero middle redundancy).
    - Extreme right dataset labels only.
    - Bottom X-axis labels: 'Premature Layer Depth (Aleatoric)' and 'Premature Layer Depth (Epistemic)'.
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    from matplotlib.lines import Line2D
    
    fig, axes = plt.subplots(3, 6, figsize=figsize, sharex=False, sharey=False)
    
    # left=0.048 adds horizontal breathing room between Y-label text and Column 0
    plt.subplots_adjust(top=0.875, bottom=0.080, left=0.048, right=0.972, hspace=0.065, wspace=0.028)
    
    row_ylimits = {
        0: (0.53, 0.77),  # TruthfulQA
        1: (0.38, 0.76),  # OpenBookQA
        2: (0.38, 0.76)   # ScienceQA (Accommodates Phi-3-Medium peak at 0.723)
    }
    row_yticks = {
        0: np.array([0.55, 0.60, 0.65, 0.70, 0.75]),
        1: np.array([0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]),
        2: np.array([0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75])
    }
    
    col_configs = [
        ('aleatoric', 'aurocs_em',   '-',  'LLaMA', 32, [1, 10, 20, 30]),
        ('aleatoric', 'aurocs_em',   '-',  'Qwen',  48, [1, 15, 30, 45]),
        ('aleatoric', 'aurocs_em',   '-',  'Phi',   48, [1, 15, 30, 45]),
        ('epistemic', 'aurocs_diff', '--', 'LLaMA', 32, [1, 10, 20, 30]),
        ('epistemic', 'aurocs_diff', '--', 'Qwen',  48, [1, 15, 30, 45]),
        ('epistemic', 'aurocs_diff', '--', 'Phi',   48, [1, 15, 30, 45])
    ]
    
    column_legend_items = {i: {} for i in range(6)}
    
    for row_idx, (ds_key, ds_title) in enumerate(TARGET_DATASETS):
        y_lim = row_ylimits[row_idx]
        y_ticks = row_yticks[row_idx]
        
        for col_idx, (m_type, m_key, l_style, fam, max_d, x_ticks) in enumerate(col_configs):
            ax = axes[row_idx, col_idx]
            
            for spine in ax.spines.values():
                spine.set_linewidth(0.65)
                spine.set_color('#444444')
                
            cell_runs = [
                r for r in sweep_data 
                if r['dataset'].lower() == ds_key.lower() and r['family'].lower() == fam.lower()
            ]
            cell_runs.sort(key=model_sort_key)
            
            if cell_runs:
                for r_idx, r_item in enumerate(cell_runs):
                    m_raw = r_item['model'].lower()
                    color = MODEL_COLORS.get(m_raw, FALLBACK_PALETTE[r_idx % len(FALLBACK_PALETTE)])
                    m_lbl = clean_model_label(r_item['model'])
                    
                    ax.plot(
                        r_item['depths'], r_item[m_key],
                        label=m_lbl,
                        linestyle=l_style,
                        linewidth=1.9,
                        color=color,
                        alpha=0.92
                    )
                    
                    if m_lbl not in column_legend_items[col_idx]:
                        proxy = Line2D([0], [0], linestyle='none', marker='o', markersize=6.0,
                                       markerfacecolor=color, markeredgecolor='none')
                        column_legend_items[col_idx][m_lbl] = proxy
                        
            ax.set_xticks(x_ticks)
            ax.set_xlim(0.5, max_d + 0.5)
            ax.set_ylim(y_lim)
            ax.set_yticks(y_ticks)
            ax.grid(True, linestyle=':', linewidth=0.55, alpha=0.35, color='#b0b0b0')
            
            ax.tick_params(
                axis='both', which='major',
                length=2.5,
                width=0.65,
                color='#555555',
                labelsize=10.5,
                pad=2.5
            )
            
            if col_idx == 0:
                ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
            else:
                ax.set_yticklabels([])
                
            if row_idx < 2:
                ax.set_xticklabels([])
                
            # Column Family Title
            if row_idx == 0:
                fam_title = FAMILY_TITLES.get(fam, f"{fam} Family")
                ax.set_title(fam_title, fontsize=12.5, fontweight='bold', pad=36)
                
            # Dataset name ONLY on extreme right column (Col 5) - shifted slightly left (x=1.018)
            if col_idx == 5:
                ax.text(1.018, 0.5, ds_title, transform=ax.transAxes,
                        va='center', ha='left', rotation=270,
                        fontsize=13.0, fontweight='bold', color='#222222')

    # --- BOTTOM X-AXIS LABELS (Aleatoric vs Epistemic blocks) ---
    fig.text(0.270, 0.018, "Premature Layer Depth (Aleatoric)",
             ha='center', va='bottom', fontsize=13.0, fontweight='bold', color='#111111')
    fig.text(0.748, 0.018, "Premature Layer Depth (Epistemic)",
             ha='center', va='bottom', fontsize=13.0, fontweight='bold', color='#111111')

    # --- SINGLE GLOBAL Y-AXIS LABEL (Spaced cleanly from Column 0) ---
    fig.text(0.012, 0.480, "Error Detection AUROC", va='center', ha='center',
             rotation='vertical', fontsize=13.5, fontweight='bold', color='#111111')

    # --- CRISP LEGENDS ---
    for col_idx in range(6):
        handles_dict = column_legend_items[col_idx]
        if handles_dict:
            top_ax = axes[0, col_idx]
            labels = list(handles_dict.keys())
            handles = list(handles_dict.values())
            top_ax.legend(
                handles, labels,
                loc='lower center',
                bbox_to_anchor=(0.5, 1.02),
                ncol=len(labels),
                fontsize=10.2,
                handlelength=0.40,
                handletextpad=0.30,
                columnspacing=0.60,
                borderpad=0.28,
                frameon=True,
                framealpha=0.95,
                edgecolor='#d0d0d0'
            )
            
    os.makedirs('figures', exist_ok=True)
    pdf_path = os.path.join('figures', 'paper_layerwise_unified_3x6.pdf')
    png_path = os.path.join('figures', 'paper_layerwise_unified_3x6.png')
    
    fig.savefig(pdf_path, format='pdf', bbox_inches='tight')
    fig.savefig(png_path, format='png', dpi=300, bbox_inches='tight')
    print('=' * 80)
    print(f"[SUCCESS] Unified 3x6 Grid Exported (Dataset Names Shifted Left):")
    print(f"  - Vector PDF : {os.path.abspath(pdf_path)}")
    print(f"  - 300 DPI PNG: {os.path.abspath(png_path)}")
    print('=' * 80)
    return fig

def plot_layerwise_3x3_grid(sweep_data, metric_type='aleatoric', figsize=(13.2, 8.4)):
    """Standalone 3x3 grid helper for individual Aleatoric or Epistemic plots."""
    import matplotlib.ticker as ticker
    from matplotlib.lines import Line2D
    
    is_aleatoric = (metric_type.lower() == 'aleatoric')
    metric_key = 'aurocs_em' if is_aleatoric else 'aurocs_diff'
    line_style = '-' if is_aleatoric else '--'
    
    fig, axes = plt.subplots(3, 3, figsize=figsize, sharex=False, sharey=False)
    plt.subplots_adjust(top=0.875, bottom=0.075, left=0.070, right=0.948, hspace=0.065, wspace=0.035)
    
    column_legend_items = {0: {}, 1: {}, 2: {}}
    
    row_ylimits = {
        0: (0.53, 0.77),
        1: (0.38, 0.76),
        2: (0.38, 0.76)
    }
    row_yticks = {
        0: np.array([0.55, 0.60, 0.65, 0.70, 0.75]),
        1: np.array([0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]),
        2: np.array([0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75])
    }
    col_xlimits = {0: 32, 1: 48, 2: 48}
    col_xticks = {0: [1, 10, 20, 30], 1: [1, 15, 30, 45], 2: [1, 15, 30, 45]}
    
    for row_idx, (ds_key, ds_title) in enumerate(TARGET_DATASETS):
        y_lim = row_ylimits[row_idx]
        y_ticks = row_yticks[row_idx]
        
        for col_idx, fam in enumerate(TARGET_FAMILIES):
            ax = axes[row_idx, col_idx]
            max_d = col_xlimits[col_idx]
            x_ticks = col_xticks[col_idx]
            
            for spine in ax.spines.values():
                spine.set_linewidth(0.65)
                spine.set_color('#444444')
            
            cell_runs = [
                r for r in sweep_data 
                if r['dataset'].lower() == ds_key.lower() and r['family'].lower() == fam.lower()
            ]
            cell_runs.sort(key=model_sort_key)
            
            if cell_runs:
                for r_idx, r_item in enumerate(cell_runs):
                    m_raw = r_item['model'].lower()
                    color = MODEL_COLORS.get(m_raw, FALLBACK_PALETTE[r_idx % len(FALLBACK_PALETTE)])
                    m_lbl = clean_model_label(r_item['model'])
                    
                    ax.plot(
                        r_item['depths'], r_item[metric_key],
                        label=m_lbl, linestyle=line_style, linewidth=2.0, color=color, alpha=0.92
                    )
                    if m_lbl not in column_legend_items[col_idx]:
                        proxy = Line2D([0], [0], linestyle='none', marker='o', markersize=6.0,
                                       markerfacecolor=color, markeredgecolor='none')
                        column_legend_items[col_idx][m_lbl] = proxy
                        
            ax.set_xticks(x_ticks)
            ax.set_xlim(0.5, max_d + 0.5)
            ax.set_ylim(y_lim)
            ax.set_yticks(y_ticks)
            ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
            ax.grid(True, linestyle=':', linewidth=0.55, alpha=0.35, color='#b0b0b0')
            ax.tick_params(axis='both', which='major', length=2.5, width=0.65, color='#555555', labelsize=10.5, pad=2.5)
            
            if col_idx > 0:
                ax.set_yticklabels([])
            if row_idx < 2:
                ax.set_xticklabels([])
            if row_idx == 0:
                fam_title = FAMILY_TITLES.get(fam, f"{fam} Family")
                ax.set_title(fam_title, fontsize=12.5, fontweight='bold', pad=36)
            if col_idx == 2:
                ax.text(1.018, 0.5, ds_title, transform=ax.transAxes,
                        va='center', ha='left', rotation=270, fontsize=13.0, fontweight='bold', color='#222222')
                        
    metric_sub = "aleatoric" if is_aleatoric else "epistemic"
    fig.text(0.505, 0.015, f"Premature Layer Depth ({metric_sub.capitalize()})",
             ha='center', va='bottom', fontsize=13.0, fontweight='bold', color='#111111')
    fig.text(0.015, 0.485, "Error Detection AUROC", va='center', ha='center',
             rotation='vertical', fontsize=13.5, fontweight='bold', color='#111111')
             
    for col_idx in range(3):
        handles_dict = column_legend_items[col_idx]
        if handles_dict:
            top_ax = axes[0, col_idx]
            labels = list(handles_dict.keys())
            handles = list(handles_dict.values())
            top_ax.legend(
                handles, labels, loc='lower center', bbox_to_anchor=(0.5, 1.02),
                ncol=len(labels), fontsize=10.2, handlelength=0.40,
                handletextpad=0.30, columnspacing=0.60, borderpad=0.28,
                frameon=True, framealpha=0.95, edgecolor='#d0d0d0'
            )
            
    os.makedirs('figures', exist_ok=True)
    out_name = f'paper_layerwise_{metric_sub}_3x3'
    fig.savefig(f'figures/{out_name}.pdf', format='pdf', bbox_inches='tight')
    fig.savefig(f'figures/{out_name}.png', format='png', dpi=300, bbox_inches='tight')
    return fig

def plot_layerwise_3x3_overlay(sweep_data, figsize=(7.2, 5.0), mark_every=5):
    """
    Final Publication-Ready 3x3 Overlay Grid (Option A Nomenclature):
    - Top metric legend aligned with paper Table 1 & Section 3.3:
      * 'Aleatoric uncertainty (H_mix)' (solid line, circle markers)
      * 'Epistemic uncertainty (ΔH)' (dashed line, square markers)
    - Vertical RHS dataset labels (TruthfulQA, OpenBookQA, ScienceQA).
    - Compact vertical spacing between top legend and column headers.
    - Fine line weights with clean hollow precision markers.
    - X-axis label: 'Layer-Window Size' (strictly no 'K').
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    from matplotlib.lines import Line2D

    target_datasets = [
        ('truthfulqa', 'TruthfulQA'),
        ('openbookqa', 'OpenBookQA'),
        ('scienceqa',  'ScienceQA')
    ]

    family_titles = {
        'LLaMA': 'LLaMA',
        'Qwen':  'Qwen-2.5',
        'Phi':   'Phi-3'
    }

    model_colors = {
        # LLaMA Family
        'llama-3.2-1b':      '#0072B2',  # Blue
        'llama-3.2-3b':      '#009E73',  # Green
        'meta-llama-3-8b':   '#D55E00',  # Orange-red
        'llama-3-8b':        '#D55E00',
        # Qwen-2.5 Family
        'qwen-1.5b':         '#0072B2',  # Blue
        'qwen2.5-1.5b':      '#0072B2',
        'qwen2.5-3b':        '#D55E00',  # Orange-red
        'qwen2.5-7b':        '#009E73',  # Green
        'qwen2.5-14b':       '#8B5AA5',  # Purple
        # Phi-3 Family
        'phi-3-mini':        '#009E73',  # Green
        'phi-3-mini-4k':     '#009E73',
        'phi-3-medium-4k':   '#8B5AA5',  # Purple
        'phi-3-medium':      '#8B5AA5'
    }
    fallback_palette = ['#0072B2', '#D55E00', '#009E73', '#8B5AA5', '#56B4E9', '#E69F00']

    def get_clean_label(m_str):
        m = str(m_str).lower()
        if '1b' in m and 'llama' in m: return '3.2-1B'
        if '3b' in m and 'llama' in m: return '3.2-3B'
        if ('8b' in m or 'llama-3-8b' in m) and 'llama' in m: return '3-8B'
        if '1.5b' in m: return '1.5B'
        if '3b' in m and 'qwen' in m: return '3B'
        if '7b' in m and 'qwen' in m: return '7B'
        if '14b' in m and 'qwen' in m: return '14B'
        if 'mini' in m: return 'Mini'
        if 'medium' in m: return 'Medium'
        return str(m_str).replace('meta-', '').replace('-instruct', '').replace('-chat', '').replace('-4k', '').upper()

    def get_sort_key(item):
        m = item['model'].lower()
        if '1b' in m or '1.5b' in m: return 1
        if '3b' in m or 'mini' in m: return 3
        if '7b' in m or '8b' in m: return 7
        if '14b' in m or 'medium' in m: return 14
        return 99

    fig, axes = plt.subplots(3, 3, figsize=figsize, sharex=False, sharey=False)
    
    # Balanced publication margins
    plt.subplots_adjust(top=0.835, bottom=0.095, left=0.088, right=0.942, hspace=0.15, wspace=0.08)

    y_lim = (0.38, 0.78)
    y_ticks = np.array([0.40, 0.50, 0.60, 0.70])

    col_configs = [
        ('LLaMA', 32, [1, 10, 20, 30]),
        ('Qwen',  48, [1, 15, 30, 45]),
        ('Phi',   48, [1, 15, 30, 45])
    ]

    family_legend_items = {'LLaMA': {}, 'Qwen': {}, 'Phi': {}}

    for row_idx, (ds_key, ds_title) in enumerate(target_datasets):
        for col_idx, (fam, max_d, x_ticks) in enumerate(col_configs):
            ax = axes[row_idx, col_idx]

            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_color('#888888')
            ax.spines['bottom'].set_color('#888888')
            ax.spines['left'].set_linewidth(0.65)
            ax.spines['bottom'].set_linewidth(0.65)

            ax.axhline(0.50, color='#888888', linestyle=':', linewidth=0.85, alpha=0.85, zorder=1)
            ax.grid(axis='y', linestyle='-', linewidth=0.45, color='#eeeeee', zorder=0)

            cell_runs = [
                r for r in sweep_data
                if r['dataset'].lower() == ds_key.lower() and r['family'].lower() == fam.lower()
            ]
            cell_runs.sort(key=get_sort_key)

            for r_idx, r_item in enumerate(cell_runs):
                m_raw = r_item['model'].lower()
                color = model_colors.get(m_raw, fallback_palette[r_idx % len(fallback_palette)])
                m_lbl = get_clean_label(r_item['model'])
                n_pts = len(r_item.get('depths', []))
                
                # Adaptive stride: maintains ~5-6 clean hollow anchor points per line
                m_stride = max(3, mark_every if n_pts >= 14 else 2)

                # 1. Aleatoric uncertainty: Solid line + Hollow circle marker ('o')
                if r_item.get('aurocs_em'):
                    ax.plot(
                        r_item['depths'], r_item['aurocs_em'],
                        linestyle='-', linewidth=1.35,
                        marker='o', markersize=3.5,
                        markerfacecolor='white', markeredgecolor=color, markeredgewidth=1.0,
                        markevery=m_stride, color=color, alpha=0.92, zorder=3
                    )

                # 2. Epistemic uncertainty: Dashed line + Hollow square marker ('s')
                if r_item.get('aurocs_diff'):
                    ax.plot(
                        r_item['depths'], r_item['aurocs_diff'],
                        linestyle=(0, (3.5, 1.8)), linewidth=1.30,
                        marker='s', markersize=3.3,
                        markerfacecolor='white', markeredgecolor=color, markeredgewidth=1.0,
                        markevery=m_stride, color=color, alpha=0.92, zorder=3
                    )

                if m_lbl not in family_legend_items[fam]:
                    proxy = Line2D([0], [0], linestyle='none', marker='o', markersize=4.5,
                                   markerfacecolor=color, markeredgecolor='none')
                    family_legend_items[fam][m_lbl] = proxy

            ax.set_xlim(0.5, max_d + 0.5)
            ax.set_ylim(y_lim)
            ax.set_xticks(x_ticks)
            ax.set_yticks(y_ticks)
            ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
            ax.tick_params(axis='both', which='major', length=2.5, width=0.6, color='#555555', labelsize=8.0, pad=3.5)

            if col_idx > 0:
                ax.set_yticklabels([])
            if row_idx < 2:
                ax.set_xticklabels([])

            # Dataset label positioned VERTICALLY on RHS (Column 2)
            if col_idx == 2:
                ax.text(1.026, 0.50, ds_title, transform=ax.transAxes,
                        va='center', ha='left', rotation=270,
                        fontsize=10.0, fontweight='bold', color='#111111')

            # Column Family Title: pad=19 places title snug above model dots
            if row_idx == 0:
                fam_title = family_titles.get(fam, fam)
                ax.set_title(fam_title, fontsize=10.5, fontweight='bold', pad=19, color='#111111')

    grid_center_x = (0.088 + 0.942) / 2  # 0.515

    # Top Metric Style Legend: Option A (Directly matches paper Table 1 & Section 3.3)
    style_dot = Line2D(
        [0], [0], linestyle='-', linewidth=1.45,
        marker='o', markersize=4.6, markerfacecolor='white', markeredgecolor='#1e293b', markeredgewidth=1.0,
        color='#1e293b', label=r'Aleatoric uncertainty ($H_{\mathrm{mix}}$)'
    )
    style_square = Line2D(
        [0], [0], linestyle=(0, (3.5, 1.8)), linewidth=1.40,
        marker='s', markersize=4.4, markerfacecolor='white', markeredgecolor='#1e293b', markeredgewidth=1.0,
        color='#1e293b', label=r'Epistemic uncertainty ($\Delta H$)'
    )
    fig.legend(
        handles=[style_dot, style_square],
        loc='upper center', bbox_to_anchor=(grid_center_x, 0.962),
        ncol=2, frameon=False, fontsize=8.2, handlelength=2.2, handletextpad=0.35, columnspacing=1.8
    )

    # Column Model Legends (sitting directly above top row subplots)
    for col_idx, (fam, _, _) in enumerate(col_configs):
        top_ax = axes[0, col_idx]
        handles_dict = family_legend_items[fam]
        if handles_dict:
            top_ax.legend(
                handles_dict.values(), handles_dict.keys(),
                loc='lower center', bbox_to_anchor=(0.5, 1.01),
                ncol=len(handles_dict), frameon=False, fontsize=7.6,
                handletextpad=0.20, columnspacing=0.55
            )

    # Global Axis Labels: strictly 'Layer-Window Size' (no 'K')
    fig.text(0.016, 0.465, 'Failure-detection AUROC', va='center', ha='left',
             rotation='vertical', fontsize=9.5, fontweight='bold', color='#111111')
    fig.text(grid_center_x, 0.018, 'Layer-Window Size', ha='center', va='bottom',
             fontsize=9.5, fontweight='bold', color='#111111')

    os.makedirs('figures', exist_ok=True)
    pdf_path = os.path.join('figures', 'paper_layerwise_3x3_overlay.pdf')
    png_path = os.path.join('figures', 'paper_layerwise_3x3_overlay.png')
    fig.savefig(pdf_path, format='pdf', bbox_inches='tight')
    fig.savefig(png_path, format='png', dpi=300, bbox_inches='tight')

    print(f'[SUCCESS] Exported Final 3x3 Overlay Grid (Option A Paper Nomenclature):\n  - Vector PDF : {os.path.abspath(pdf_path)}\n  - 300 DPI PNG: {os.path.abspath(png_path)}')
    return fig



### 3. Publication Unified $3 \times 6$ Grid Visualization (Aleatoric & Epistemic)

Renders the complete side-by-side $3 \times 6$ publication figure combining Aleatoric ($H_{\text{mix}}$) and Epistemic ($\Delta H$) sweeps.


In [ ]:
# Render and Export Publication Figure (Option A: Aleatoric vs Epistemic Uncertainty)
fig_3x3 = plot_layerwise_3x3_overlay(sweep_results, mark_every=5)


### 3.1 Optional: Individual Standalone $3 \times 3$ Grids

Exports standalone 3x3 grids for individual analysis if needed.


In [ ]:
# fig_au = plot_layerwise_3x3_grid(sweep_results, metric_type='aleatoric')
# fig_eu = plot_layerwise_3x3_grid(sweep_results, metric_type='epistemic')


## 4. Peak AUROC & Optimal Premature Ensembling Depth ($K_a^*$)

Summarizes the optimal layer ensembling depth $K_a^*$ (for Aleatoric $H_{\text{mix}}$) and horizon $K_e^*$ (for Epistemic $\Delta H$) alongside task accuracy at BLEURT threshold $\geq 0.1$.


In [ ]:
summary_rows = []
for r in sweep_results:
    if not r['aurocs_em']:
        continue
    best_idx_em = int(np.argmax(r['aurocs_em']))
    best_ka = r['depths'][best_idx_em]
    peak_auroc_em = r['aurocs_em'][best_idx_em]
    
    best_idx_diff = int(np.argmax(r['aurocs_diff']))
    best_ke = r['depths'][best_idx_diff]
    peak_auroc_diff = r['aurocs_diff'][best_idx_diff]
    
    summary_rows.append({
        'Dataset': r['dataset'].upper(),
        'Family': r['family'],
        'Model': clean_model_label(r['model']),
        'Task Accuracy': f"{r['accuracy']*100:.1f}%",
        'Optimal Ka* (AU)': best_ka,
        'Peak AUROC (H_mix)': f"{peak_auroc_em:.4f}",
        'Optimal Ke* (EU)': best_ke,
        'Peak AUROC (Delta H)': f"{peak_auroc_diff:.4f}"
    })

if summary_rows:
    df_summary = pd.DataFrame(summary_rows)
    df_summary = df_summary.sort_values(by=['Dataset', 'Family', 'Model']).reset_index(drop=True)
    print('TABLE: OPTIMAL LAYER ENSEMBLING DEPTHS & PEAK AUROC (BLEURT >= 0.1)')
    print('=' * 100)
    display(df_summary)
else:
    print('No summary rows available.')
